In [54]:
import polars as pl
import altair

In [ ]:
"""
Exploration conclusions:
    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
        
        Example:
        
        [contract_desc]                     [nip]
        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
        
        Decisions:
        
        There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis. 
"""


'\nExploration conclusions:\n    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings\n\n        Example:\n\n        [contract_desc]                     [nip]\n        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"\n'

In [61]:
df = pl.read_excel(r"data\umk_contracts_202*.xlsx")

df = df.rename({
    'Lp': 'idx',
    'rok_plik': 'year_filename',
    'Numer w rejestrze': 'registry',
    'Nazwa Kontrahenta ': 'contractor_name',
    'NIP': 'nip',
    'Kwota złotych brutto': 'gross_total_pln',
    'Data obowiązywania od': 'date_valid_from',
    'Data obowiązywania do': 'date_valid_to',
    'Data zawarcia': 'date_contract_signed',
    'Przedmiot umowy': 'contract_desc',
    'Jedn. Realizująca': 'administrative_unit'
})

from types import SimpleNamespace

_names = SimpleNamespace({col: col for col in df.columns})

C:\Users\Manager\AppData\Local\Temp\ipykernel_20504\1031874544.py:1: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(r"data\umk_contracts_202*.xlsx")


In [ ]:
"""
1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
    
    Example:
    
    [contract_desc]                     [nip]
    "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
    
    Decisions:
    
    There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis. 
"""

df = (
    df
    .select(
        pl.col(_names.idx, _names.contractor_name, _names.nip)
    )
    .with_columns(
        pl.col(_names.nip).str.contains(',').alias("contains_comma")
    )
    # .select(
    #     pl.col("contains_comma").sum()
    # )
    .filter(
        pl.col("contains_comma")
    )
    .with_columns(
        pl.col(_names.contractor_name).str.split(','),
        pl.col(_names.nip).str.split(',')
    )
    .with_columns(
        abs(pl.col(_names.contractor_name).list.len().cast(pl.Int64) - pl.col(_names.nip).list.len().cast(pl.Int64)).alias("diff_len") #cast to Int64 or we get underflow
    )
    .filter(
        pl.col("diff_len") != 0
    )
)


idx,year_filename,registry,contractor_name,nip,gross_total_pln,date_valid_from,date_valid_to,date_contract_signed,contract_desc,administrative_unit
i64,i64,str,str,str,f64,str,str,str,str,str
1,2023,"""-W/IV-B/291/AU/4/2023""","""TRANSLATION STREET JASIUKIEWICZ - SZCZUREK SP.Z O.…","""6793172270""",1476.0,"""2023-10-03""","""2023-10-04""","""2023-10-03""","""USŁUGA TŁUMACZENIA KONSEKUTYWNEGO""","""AU"""
2,2023,"""-W/IV-B/276/AU/3/2023""","""POLSKA PRESS SP. Z O.O.""","""5220103609""",8906.8,"""2023-09-18""","""2023-12-20""","""2023-09-18""","""PUBLIKACJA NA ZLECENIE OBWIESZCZEŃ W PRASIE CODZIE…","""AU"""
3,2023,"""-W/IV-B/20/AU/1/2023""","""AGORA S.A.""","""5260305644""",5868.58,"""2023-01-19""","""2023-02-03""","""2023-01-19""","""PUDLIKACJA ODWIESZCZEŃ W PRASIE CODZIENNEJ O ZASIE…","""AU"""
4,2023,"""-W/IV-B/129/AU/2/2023""","""PBSERVICE S.C KAMIL WYKROTA, KRZYSZTOF PAWŁOWSKI""","""5833149869""",8988.84,"""2023-04-26""","""2023-05-31""","""2023-04-26""","""WYKONANIE KOPERT Z NADRUKIEM WRAZ Z DOSTAWĄ""","""AU"""
5,2023,"""-W/I/553/AU/1/2023""","""PBSERVICE S.C KAMIL WYKROTA, KRZYSZTOF PAWŁOWSKI""","""5833149869""",12839.97,"""2023-02-01""","""2023-12-31""","""2023-01-31""","""DZIERŻAWA UŻĄDZENI AKOPERTUJĄCEGO PITNEY BOWES DI3…","""AU"""
…,…,…,…,…,…,…,…,…,…,…
5993,2025,"""-W/XII/3/WT/2/2025""","""MIĘDZYNARODOWE CENTRUM KULTURY""","""6751000104""",18450.0,"""2025-03-07""","""2025-05-11""","""2025-03-07""","""WYNAJEM PRZESTRZENI KONFERENCYJNEJ WRAZ Z ZAPLECZE…","""WT"""
5994,2025,"""-W/XII/2/WT/1/2025""",""" KRAKÓW5020 SP. Z O.O. W LIKWIDACJI, WINDMILL SPÓ…","""6762609702, 1251629500""",48093.0,"""2025-02-11""","""2025-02-17""","""2025-02-11""","""WYNAJEM PRZEZ GMK ORAZ AGENCJĘ WINDMILL DZIAŁAJĄCĄ…","""WT"""
5995,2025,"""-W/XII/30/WT/13/2025""",""" AGENCJA ROZWOJU MIASTA KRAKOWA SPÓŁKA Z OGRANICZ…","""6751698978, 9452263458""",36592.5,"""2025-10-10""","""2025-11-15""","""2025-10-10""","""WYNAJEM W DNIU 15 LISTOPADA 2025 R. PRZESTRZENI CE…","""WT"""


In [66]:
df_temp = df #[200:300]
df_temp

idx,year_filename,registry,contractor_name,nip,gross_total_pln,date_valid_from,date_valid_to,date_contract_signed,contract_desc,administrative_unit
i64,i64,str,str,str,f64,str,str,str,str,str
1,2023,"""-W/IV-B/291/AU/4/2023""","""TRANSLATION STREET JASIUKIEWICZ - SZCZUREK SP.Z O.…","""6793172270""",1476.0,"""2023-10-03""","""2023-10-04""","""2023-10-03""","""USŁUGA TŁUMACZENIA KONSEKUTYWNEGO""","""AU"""
2,2023,"""-W/IV-B/276/AU/3/2023""","""POLSKA PRESS SP. Z O.O.""","""5220103609""",8906.8,"""2023-09-18""","""2023-12-20""","""2023-09-18""","""PUBLIKACJA NA ZLECENIE OBWIESZCZEŃ W PRASIE CODZIE…","""AU"""
3,2023,"""-W/IV-B/20/AU/1/2023""","""AGORA S.A.""","""5260305644""",5868.58,"""2023-01-19""","""2023-02-03""","""2023-01-19""","""PUDLIKACJA ODWIESZCZEŃ W PRASIE CODZIENNEJ O ZASIE…","""AU"""
4,2023,"""-W/IV-B/129/AU/2/2023""","""PBSERVICE S.C KAMIL WYKROTA, KRZYSZTOF PAWŁOWSKI""","""5833149869""",8988.84,"""2023-04-26""","""2023-05-31""","""2023-04-26""","""WYKONANIE KOPERT Z NADRUKIEM WRAZ Z DOSTAWĄ""","""AU"""
5,2023,"""-W/I/553/AU/1/2023""","""PBSERVICE S.C KAMIL WYKROTA, KRZYSZTOF PAWŁOWSKI""","""5833149869""",12839.97,"""2023-02-01""","""2023-12-31""","""2023-01-31""","""DZIERŻAWA UŻĄDZENI AKOPERTUJĄCEGO PITNEY BOWES DI3…","""AU"""
…,…,…,…,…,…,…,…,…,…,…
5993,2025,"""-W/XII/3/WT/2/2025""","""MIĘDZYNARODOWE CENTRUM KULTURY""","""6751000104""",18450.0,"""2025-03-07""","""2025-05-11""","""2025-03-07""","""WYNAJEM PRZESTRZENI KONFERENCYJNEJ WRAZ Z ZAPLECZE…","""WT"""
5994,2025,"""-W/XII/2/WT/1/2025""",""" KRAKÓW5020 SP. Z O.O. W LIKWIDACJI, WINDMILL SPÓ…","""6762609702, 1251629500""",48093.0,"""2025-02-11""","""2025-02-17""","""2025-02-11""","""WYNAJEM PRZEZ GMK ORAZ AGENCJĘ WINDMILL DZIAŁAJĄCĄ…","""WT"""
5995,2025,"""-W/XII/30/WT/13/2025""",""" AGENCJA ROZWOJU MIASTA KRAKOWA SPÓŁKA Z OGRANICZ…","""6751698978, 9452263458""",36592.5,"""2025-10-10""","""2025-11-15""","""2025-10-10""","""WYNAJEM W DNIU 15 LISTOPADA 2025 R. PRZESTRZENI CE…","""WT"""


In [94]:
pl.Config.set_fmt_str_lengths(300)
pl.Config.set_tbl_width_chars(100)

(
    df_temp
    # .select(
    #     pl.col(_names.idx, _names.contractor_name, _names.nip)
    # )
    .with_columns(
        pl.col(_names.nip).str.contains('-').alias("contains_comma")
    )
    .filter(
        pl.col("contains_comma")
    )
    # .with_columns(
    #     pl.col(_names.contractor_name).str.split(','),
    #     pl.col(_names.nip).str.split(',')
    # )
    # .with_columns(
    #     pl.col(_names.contractor_name).list.len().cast(pl.Int64).alias("contractor_name_len"),
    #     pl.col(_names.nip).list.len().cast(pl.Int64).alias("nip_len"),
    #     abs(pl.col(_names.contractor_name).list.len().cast(pl.Int64) - pl.col(_names.nip).list.len().cast(pl.Int64)).alias("diff_len") #cast to Int64 or we get underflow
    # )
    # .filter(
    #     (pl.col("diff_len") != 0)
    #     & (pl.col("contractor_name_len") > pl.col("nip_len"))
    #     # & (pl.col("nip_len") == 2)
    # )
)



idx,year_filename,registry,contractor_name,nip,gross_total_pln,date_valid_from,date_valid_to,date_contract_signed,contract_desc,administrative_unit,contains_comma
i64,i64,str,str,str,f64,str,str,str,str,str,bool
20,2023,"""-W/I/958/BD/27/2023""","""WSZOŁEK GABRIELA""","""-""",17000.0,"""2023-03-08""","""2023-11-30""","""2023-03-08""","""REDAKCJA 7 NUMERÓW GAZETKI LOKALNEJ DZIENICY XI PODGÓRZE DUCHACKIE W 2023 R.""","""BD""",true
22,2023,"""-W/I/863/BD/25/2023""","""GAWRON JAKUB""","""-""",15000.0,"""2023-02-27""","""2023-12-13""","""2023-02-27""","""OPRACOWANIE REDAKCYJNE 7 NUMERÓW GAZETKI RADY I ZARZĄDU DZIELNICY IX ŁAGIEWNIKI - BOREK FAŁĘCKI,.""","""BD""",true
25,2023,"""-W/I/828/BD/22/2023""","""SZAFLARSKA ANNA""","""-""",4500.0,"""2023-03-01""","""2023-11-30""","""2023-03-01""","""OBSŁUGA TABLIC INFORMACYJNYCH RADY DZIELNICY XI PODGÓRZE DUCHACKIE.""","""BD""",true
27,2023,"""-W/I/806/BD/20/2023""","""CZAJA DAWID""","""-""",9999.0,"""2023-02-15""","""2023-12-15""","""2023-02-15""","""AKTUALIZACJA INFORMACJI W MEDIACH SPOŁECZNOŚCIOWYCH ORAZ REDAGOWANIE STRONY INTERNETOWEJ DZIELNICY VIII DĘBNIKI.""","""BD""",true
28,2023,"""-W/I/792/BD/19/2023""","""CZAJA DAWID""","""-""",2992.0,"""2023-02-15""","""2023-12-15""","""2023-02-15""","""OBSŁUGA TABLIC INFORMACYJNYCH RADY DZIELNICY VIII DĘBNIKI""","""BD""",true
…,…,…,…,…,…,…,…,…,…,…,…
5922,2025,"""-W/I/767/WT/7/2025""","""ZMYŚLONY PIOTR""","""-""",20000.0,"""2025-02-21""","""2025-02-28""","""2025-02-21""","""PRZYGOTOWANIE AUTORSKIEJ KONCEPCJI MERYTORYCZNEJ I ZAŁOŻEŃ PROGRAMOWYCH KONFERENCJI ""MIASTA HISTORYCZNE 3.0"",""","""WT""",true
5971,2025,"""-W/I/3084/WT/46/2025""","""WORLDWIDE EVENTS GROUP LTD""","""-""",105319.48,"""2025-08-06""","""2026-09-26""","""2025-08-06""","""UDZIAŁ W TARGACH PRZEDSTAWICIELI KRAKOWSKIEGO BIURA KONGRESÓW/KRAKOW CONVENTION BUREAU PROMUJĄCYCH KRAKÓW I MAŁOPOLSKĘ JAKO DESTYNACJĘ MICE""","""WT""",true
5972,2025,"""-W/IV-B/164/WT/3/2025""","""CITY DESTINATIONS ALLIANCE""","""-""",7877.01,"""2025-06-30""","""2025-08-29""","""2025-06-25""","""USŁUGA WSTĘPU (W RAMACH TZW. OPŁATY KONFERENCYJNEJ) DLA P. JAKUBA CHMIELNICKIEGO - PRACOWNIKA WYDZIAŁU DS. TURYSTYKI NA WYDARZENIE CITYDNA SUMMER SCHOOL 2025 W LUKSEMBURGU.W DNIACH 25-29SIERPNIA 2025""","""WT""",true


In [65]:
df_temp.filter(pl.col("idx") == 3865)

idx,year_filename,registry,contractor_name,nip,gross_total_pln,date_valid_from,date_valid_to,date_contract_signed,contract_desc,administrative_unit
i64,i64,str,str,str,f64,str,str,str,str,str
3865,2023,"""-W/V/52/SZ/31/2023""",""" KRAKOWSKA AKADEMIA IM. ANDRZEJA FRYCZA MODRZEWSK…","""6762134096, 6750001923, 6750006257, -, -, 67610119…",0.0,"""2023-04-20""",null,"""2023-04-20""","""WPÓŁPRACA W OBSZARZE OSÓB NIEPEŁNOSPRAWNYCH""","""SZ"""
3865,2024,"""-W/II/156/OU/9/2024""","""ZAKŁADELEKTROINSTALACJESTOPKAKRZYSZTOF""","""5511200091""",18696.0,"""2024-07-15""","""2024-08-30""","""2024-07-12""","""WYKONANIEWYMIANYWEWNĘTRZNEJLINIZASILAJĄCEJDEDYKOWA…","""OU"""
3865,2025,"""-W/I/2407/SP/149/2025""","""FUNDACJA PIŁKARSKI KRAKÓW""","""6762611774""",917500.0,"""2025-05-20""","""2025-12-19""","""2025-05-20""","""REALIZACJA ZADANIA PUBLICZNEGO PM: ""PIŁKARSKIE MAR…","""SP"""
